In [1]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

WORKING_DIR = pc.working_dir
PATIENT_DATA_DIR = WORKING_DIR / "patient_data"

SHRINK_FRACTIONS = [0.05, 0.10, 0.175, 0.20]
SHRINK_COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"]
SHRINK_LABELS = ["5%", "10%", "17.5%", "20%"]

OUTPUT_DIR = Path("shrink_fraction_images")
OUTPUT_DIR.mkdir(exist_ok=True)

splits_df = pd.read_csv(_PROJECT_ROOT / "splits" / "splits_01-15-26.csv")
patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Patients: {len(patients_df)}")
print(patients_df["split"].value_counts())

Found project root at: /home/ayeluru/vascular-superenhancement-4d-flow
Patients: 215
split
train         165
test           28
validation     22
Name: count, dtype: int64


In [2]:
def load_volume(nifti_path: Path) -> np.ndarray:
    return nib.load(str(nifti_path)).get_fdata(dtype=np.float32)


def get_mid_slices(vol: np.ndarray):
    """Return (axial, coronal, sagittal) mid-slices and their plane dimensions.

    vol shape: (X, Y, Z).
    Returns tuples of (slice_2d, (dim0_size, dim1_size)) for each plane.
    """
    X, Y, Z = vol.shape
    axial    = vol[:, :, Z // 2]            # dims shown: X, Y
    coronal  = vol[:, Y // 2, :]            # dims shown: X, Z
    sagittal = vol[X // 2, :, :]            # dims shown: Y, Z
    return [
        (axial,    (X, Y), "Axial (mid-Z)"),
        (coronal,  (X, Z), "Coronal (mid-Y)"),
        (sagittal, (Y, Z), "Sagittal (mid-X)"),
    ]


def draw_shrink_boxes(ax, dim0: int, dim1: int):
    """Draw nested rectangles for each shrink fraction.

    After imshow(slice.T), horizontal axis = d0, vertical axis = d1.
    """
    for frac, color, label in zip(SHRINK_FRACTIONS, SHRINK_COLORS, SHRINK_LABELS):
        margin0 = int(dim0 * frac)
        margin1 = int(dim1 * frac)
        rect = patches.Rectangle(
            (margin0 - 0.5, margin1 - 0.5),
            dim0 - 2 * margin0,
            dim1 - 2 * margin1,
            linewidth=2,
            edgecolor=color,
            facecolor="none",
            label=label,
        )
        ax.add_patch(rect)

In [3]:
for _, row in patients_df.iterrows():
    pid = row["patient_id"]
    split = row["split"]

    nifti_dir = PATIENT_DATA_DIR / pid / "nifti"
    mag_corr_fov_dir = nifti_dir / f"4d_flow_mag_{pid}_per_timepoint_corr_fov"

    if not mag_corr_fov_dir.exists():
        print(f"  SKIP {pid}: no mag corr_fov dir")
        continue

    mag_files = sorted(mag_corr_fov_dir.glob("*.nii.gz"))
    if not mag_files:
        print(f"  SKIP {pid}: no mag files")
        continue

    # Mean magnitude across timepoints (same as what the masking code uses)
    mag_vol = np.mean(
        np.stack([load_volume(f) for f in mag_files], axis=-1),
        axis=-1,
    )
    shape = mag_vol.shape  # (X, Y, Z)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(
        f"{pid}  ({split})  —  shape {shape[0]}×{shape[1]}×{shape[2]}",
        fontsize=14,
        fontweight="bold",
    )

    slices = get_mid_slices(mag_vol)

    for ax, (slice_2d, (d0, d1), title) in zip(axes, slices):
        ax.imshow(
            slice_2d.T,
            origin="lower",
            cmap="gray",
            vmin=0,
            vmax=np.percentile(mag_vol, 99),
            aspect="equal",
        )
        draw_shrink_boxes(ax, d0, d1)
        ax.set_title(f"{title}\n{d0}×{d1}", fontsize=11)
        ax.set_xlabel(f"dim1 ({d1} vox)")
        ax.set_ylabel(f"dim0 ({d0} vox)")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="lower center",
        ncol=len(SHRINK_FRACTIONS),
        fontsize=11,
        frameon=True,
    )

    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    fig.savefig(OUTPUT_DIR / f"{split}_{pid}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  {pid} OK")

print("Done.")

  Balboloop OK
  Biswifo OK
  Bomatog OK
  Boochuto OK
  Boumorim OK
  Bovutou OK
  Cadotueg OK
  Detodu OK
  Diecudey OK
  Diepami OK
  Diequipi OK
  Dithigog OK
  Dublafer OK
  Dujomal OK
  Elagieg OK
  Golotag OK
  Grequafie OK
  Gueshifa OK
  Kuquelok OK
  Oduskueb OK
  Quetode OK
  Runusath OK
  Sepigoo OK
  Stonscuetof OK
  Suquepog OK
  Tercippun OK
  Tiepolem OK
  Tisupey OK
  Amifer OK
  Aruborn OK
  Asonlig OK
  Badiswu OK
  Bibathot OK
  Bogeebo OK
  Boudubat OK
  Bukrukesh OK
  Burapo OK
  Butiswu OK
  Cadedag OK
  Cefaru OK
  Cemuquey OK
  Ceriba OK
  Ceyebum OK
  SKIP Coosimo: no mag corr_fov dir
  Cornuefor OK
  Crutaswo OK
  Dalibul OK
  Dapafem OK
  Datokif OK
  Desoomi OK
  Diboscey OK
  Difresa OK
  Dinaspig OK
  Drusdinut OK
  Dudoblo OK
  Duquestank OK
  Dutungub OK
  Edengat OK
  Egumud OK
  Ekotey OK
  Emalem OK
  Epcedin OK
  Ernegur OK
  Erusar OK
  Eyostoy OK
  Fibrefob OK
  Fliesiemo OK
  Frahidiel OK
  Fudoquo OK
  Fuekeswa OK
  Fumtufoos OK
  Geedefou OK
  